# FAISS And Approximate Nearest Neighbor Indexes

Exhaustively scoring every stored vector is simple and accurate, but it
becomes expensive as collections grow. [FAISS](https://faiss.ai/) is a
library for efficient similarity search over dense vectors; this notebook
uses [NumPy](https://numpy.org/doc/stable/) arrays as its vector boundary.
It builds both an exact index and an
[approximate nearest neighbor (ANN)](https://arxiv.org/abs/1702.08734)
index so that the accuracy-speed trade-off is observable rather than
merely asserted.

## Learning Objectives

By the end of this notebook, you should be able to:

- add normalized chunk vectors to a FAISS index,
- map returned vector positions back to source metadata,
- configure an inverted-file (IVF) ANN index and its `nprobe` setting,
- compare approximate results against an exact-search reference.

## Why An Index Exists

```mermaid
graph LR
    A["Embedded chunks"] --> B["FAISS index"]
    B --> C["Exact scan: IndexFlatIP"]
    B --> D["ANN partition: IndexIVFFlat"]
    Q["Embedded question"] --> C
    Q --> D
    C --> E["Reference top-k"]
    D --> F["Faster candidate top-k"]
    E --> G["Compare recall"]
    F --> G
```

Dense embeddings fill most dimensions with values, as illustrated below.
FAISS stores numeric dense vectors and focuses on how to search them
quickly; it does not create semantic meaning itself. Sparse lexical
retrieval and dense semantic retrieval may also be combined in hybrid
systems, but this notebook indexes dense vectors only.

![Sentence represented as a dense vector](assets/02-dense-vector.png)

Source: [Qdrant - What is a Vector Database?](https://qdrant.tech/articles/what-is-a-vector-database/)


## Space Partitions And Approximation

An IVF index learns centroids, assigns stored vectors to nearby lists, and
searches only selected lists at query time. `nprobe` controls how many
lists are visited: a small value searches less work but may miss a
relevant neighbor; searching every list recovers the exhaustive behavior.

![Voronoi space partitioning](assets/03-voronoi-partitioning.png)

Source: [Wikimedia Commons - Voronoi diagram](https://commons.wikimedia.org/wiki/File:Voronoi_diagram.png)


In [ ]:
import hashlib
import json
import re
import time
from pathlib import Path
from typing import Protocol, cast

import faiss
import numpy as np

documents = json.loads((Path("data") / "course_handbook.json").read_text())
chunks = [
    {
        "chunk_id": f"{doc['doc_id']}-whole",
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "text": doc["text"],
    }
    for doc in documents
]
print(f"Ready to index {len(chunks)} chunks.")

The local embedding interface is repeated here so this notebook runs independently. Replace it with a trained embedding model in a production experiment, then reevaluate the index results.


In [ ]:
SYNONYMS = {
    "reimbursement": "refund",
    "reimburse": "refund",
    "charged": "payment",
    "login": "password",
    "signin": "password",
    "phone": "device",
    "throttle": "limit",
    "slow": "timeout",
}
STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "can",
    "do",
    "for",
    "from",
    "how",
    "i",
    "in",
    "is",
    "my",
    "of",
    "should",
    "the",
    "to",
    "what",
    "when",
    "with",
}

Apply the explicit baseline vocabulary policy before index construction. In a production system, this hand-written normalization layer would usually give way to an evaluated trained embedding model.


In [ ]:
def tokens(text: str) -> list[str]:
    # Match small domain paraphrases before vectors are indexed.
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [SYNONYMS.get(word, word) for word in words if word not in STOPWORDS]

In [ ]:
class HashEmbeddingModel:
    """Produce deterministic unit vectors for index demonstrations."""

    def __init__(self, dimensions: int = 2048) -> None:
        self.dimensions = dimensions

    def encode(self, texts: list[str]) -> np.ndarray:
        # FAISS expects one float32 vector row for each stored text.
        vectors = np.zeros((len(texts), self.dimensions), dtype="float32")
        for row, text in enumerate(texts):
            for token in tokens(text):
                # Stable token slots keep the local baseline reproducible.
                slot = int.from_bytes(
                    hashlib.sha256(token.encode()).digest()[:4], "little"
                )
                vectors[row, slot % self.dimensions] += 1
        # With normalized rows, inner-product search measures cosine similarity.
        faiss.normalize_L2(vectors)
        return vectors


embedder = HashEmbeddingModel()
vectors = embedder.encode([chunk["text"] for chunk in chunks])

## Exact Similarity Search With `IndexFlatIP`

For normalized vectors, inner product equals cosine similarity.
`IndexFlatIP` stores every vector and compares every one at query time.
That makes it a trustworthy reference for evaluating approximate indexes.

FAISS installs a NumPy-friendly Python interface over lower-level generated
bindings when it is imported. The small `VectorIndex` protocol below names
the runtime operations used here: add a matrix and search with a matrix.


In [ ]:
class VectorIndex(Protocol):
    """The NumPy-array interface installed by FAISS's Python wrapper."""

    def add(self, vectors: np.ndarray) -> None: ...
    def search(
        self, queries: np.ndarray, top_k: int
    ) -> tuple[np.ndarray, np.ndarray]: ...


class TrainableVectorIndex(VectorIndex, Protocol):
    nprobe: int
    ntotal: int

    def train(self, vectors: np.ndarray) -> None: ...


# Cast to the public array interface installed by the FAISS Python wrapper.
dimension = vectors.shape[1]
exact_index = cast(VectorIndex, faiss.IndexFlatIP(dimension))
exact_index.add(vectors)

The query helper restores the source metadata that FAISS positions alone cannot provide. This separation is what later makes citations and debugging possible.


In [ ]:
def retrieve(
    index: VectorIndex, question: str, top_k: int = 3
) -> list[dict[str, object]]:
    """Query a FAISS index and restore source metadata for valid IDs."""

    # FAISS returns positions; metadata remains in the parallel chunk list.
    scores, positions = index.search(embedder.encode([question]), top_k)
    return [
        {**chunks[position], "score": round(float(score), 4)}
        for score, position in zip(scores[0], positions[0])
        if position >= 0
    ]

In [ ]:
question = "The API replied with 429. How should my client retry?"
exact_results = retrieve(exact_index, question)
for rank, result in enumerate(exact_results, start=1):
    print(rank, result["doc_id"], result["score"], result["title"])

The index returns numeric positions, not business metadata. Keeping the
`chunks` metadata mapping alongside the vectors is essential for citation,
filtering, debugging, and deletion workflows in a real vector store. Notice
that requesting three results can also return zero-score neighbors: `top_k`
is a ranking limit, not a relevance guarantee. The RAG pipeline therefore
adds an evidence threshold before generation.


## Build An IVF Approximate Index

IVF needs representative vectors to learn its centroids. Twelve policies
are too small for meaningful centroid training, so this classroom example
creates small deterministic perturbations only for training the partition.
The stored searchable vectors remain the twelve actual policy chunks. In a
real application, train on a representative sample of actual embeddings.


In [ ]:
# IVF needs enough training points to learn partitions in this tiny corpus.
random = np.random.default_rng(seed=7)
training_batches = []
for _ in range(12):
    # Slight perturbations expand training data without changing stored evidence.
    noisy = vectors + random.normal(0, 0.002, vectors.shape).astype("float32")
    faiss.normalize_L2(noisy)
    training_batches.append(noisy)
training_vectors = np.vstack(training_batches)
print("Training vectors:", training_vectors.shape)

In [ ]:
number_of_lists = 3
quantizer = faiss.IndexFlatIP(dimension)
raw_ann_index = faiss.IndexIVFFlat(
    quantizer, dimension, number_of_lists, faiss.METRIC_INNER_PRODUCT
)
ann_index = cast(TrainableVectorIndex, raw_ann_index)
ann_index.train(training_vectors)
ann_index.add(vectors)
# Start with one probed list to make approximate misses observable.
ann_index.nprobe = 1
print("ANN vectors stored:", ann_index.ntotal, "lists searched:", ann_index.nprobe)

## Observe The Accuracy Trade-Off

On this tiny corpus, wall-clock latency is not meaningful; the experiment
is about result agreement. Treat exact search as the reference set and ask
how many of its top results are found by the approximate index.


In [ ]:
benchmark_questions = [
    "I need a refund for a duplicate payment.",
    "I changed phones and cannot use my second factor.",
    "What should an API client do after status 429?",
    "Can I place personal email addresses in model prompts?",
    "How do I request a project extension?",
]


def top_k_agreement(question: str, nprobe: int, top_k: int = 2) -> float:
    ann_index.nprobe = nprobe
    # Treat exhaustive search as the reference set for ANN agreement.
    expected = {item["doc_id"] for item in retrieve(exact_index, question, top_k)}
    observed = {item["doc_id"] for item in retrieve(ann_index, question, top_k)}
    return len(expected & observed) / top_k

In [ ]:
for nprobe in [1, 2, 3]:
    agreements = [top_k_agreement(question, nprobe) for question in benchmark_questions]
    print(f"nprobe={nprobe}: mean top-2 agreement={np.mean(agreements):.2f}")

When `nprobe` equals the number of lists, IVF examines all partitions and
should agree closely with exact search. With millions of vectors, the
useful operating point is usually smaller and must be selected by measuring
retrieval quality and latency together.


## A Small Timing Harness

Timings on twelve records do not represent production performance, but
knowing how to measure repeated queries prepares you for a realistic index
benchmark.


In [ ]:
def elapsed_ms(index: VectorIndex, repeated_questions: list[str]) -> float:
    # Repeat identical work so small index overhead is measurable locally.
    start = time.perf_counter()
    for item in repeated_questions:
        retrieve(index, item, top_k=3)
    return (time.perf_counter() - start) * 1000


repeated = benchmark_questions * 100
ann_index.nprobe = 1
print("Exact milliseconds:", round(elapsed_ms(exact_index, repeated), 2))
print("IVF milliseconds:", round(elapsed_ms(ann_index, repeated), 2))

Do not over-interpret these numbers: at this scale, Python calls and
notebook timing noise dominate. The meaningful result here is that you now
have an evaluation harness that can be rerun on a larger corpus.


## Index Families And Their Trade-Offs

FAISS provides several index families; managed vector databases expose
related choices behind configuration APIs.

| Index idea | Search behavior | Typical trade-off |
| --- | --- | --- |
| Flat exact search | visits every vector | accurate baseline, slower at scale |
| IVF | visits selected learned partitions | tune recall with `nprobe` |
| HNSW graph | traverses neighbor links | strong recall, memory overhead |
| Product quantization (PQ) | compresses vectors | lower memory, possible accuracy loss |

Choose an index only after defining a quality target and query workload.
ANN is useful because it spends less search work, not because approximate
answers are acceptable by default.


## Index Persistence And Metadata Lifecycle

Vector positions are meaningful only while their metadata mapping remains
synchronized. FAISS can serialize an index, but an application must also
persist source IDs, versions, access rules, and the embedding-model
identity. Re-embedding documents with a different model without rebuilding
the index makes scores uninterpretable.


In [ ]:
# An index artifact is reproducible only alongside its vector/chunk contract.
index_manifest = {
    "index_type": "IndexIVFFlat",
    "dimensions": int(dimension),
    "vector_count": int(ann_index.ntotal),
    "embedding_baseline": "deterministic_hash_v1",
    "chunk_ids": [item["chunk_id"] for item in chunks],
    "nlist": number_of_lists,
}
print(json.dumps(index_manifest, indent=2))

The manifest is not a full vector database, but it records the contracts
needed to reproduce this index. Production stores also need update and
deletion behavior: when a policy changes, stale vectors must not remain
retrievable.


## Search Quality Is Query-Dependent

An ANN setting that agrees with exact search for easy keyword-rich queries
may fail for paraphrases or closely related policies. Add a hard case
before finalizing a setting, and report it separately rather than hiding it
inside a mean score.


In [ ]:
difficult_questions = [
    "My access proof device disappeared after upgrading my hardware.",
    "Two transfers appear to cover one billing notice.",
]

for question in difficult_questions:
    ann_index.nprobe = 1
    approximate = [item["doc_id"] for item in retrieve(ann_index, question, top_k=2)]
    exact = [item["doc_id"] for item in retrieve(exact_index, question, top_k=2)]
    print(question, "\n exact:", exact, "\n approximate:", approximate)

## Exercise - Select An ANN Setting

Use the benchmark questions and choose the smallest `nprobe` value that
reaches the best measured mean top-2 agreement with exact search. Print
both the chosen setting and its agreement, then retrieve the top two
rate-limit results with that setting.


## Correction


In [ ]:
scored_settings = []
for nprobe in range(1, number_of_lists + 1):
    score = np.mean(
        [top_k_agreement(question, nprobe) for question in benchmark_questions]
    )
    scored_settings.append((nprobe, score))

# Select the least search effort that achieves the measured best agreement.
best_agreement = max(score for _, score in scored_settings)
selected_nprobe = min(
    nprobe for nprobe, score in scored_settings if score == best_agreement
)
ann_index.nprobe = selected_nprobe
print(
    "Selected nprobe:", selected_nprobe, "agreement:", round(float(best_agreement), 2)
)
for result in retrieve(ann_index, "What follows an API rate limit response?", top_k=2):
    print(result["doc_id"], result["score"], result["title"])

**Reference answer:** Select the smallest setting with the best observed
agreement and record any remaining gap rather than pretending ANN is
lossless. If no approximate setting meets the required quality threshold,
keep exact search for this workload or evaluate another index design. In
production, add latency, memory, and a larger labeled query set.


## Further Reading

- [FAISS documentation](https://faiss.ai/)
- Johnson et al., [Billion-scale similarity search with GPUs](https://arxiv.org/abs/1702.08734)
- Malkov and Yashunin, [HNSW](https://arxiv.org/abs/1603.09320)
- [Qdrant indexing concepts](https://qdrant.tech/documentation/concepts/indexing/)
